# 03 - Embeddings

Generate semantic embeddings for resumes and job descriptions using SentenceTransformer (BERT-based).

In [ ]:
!pip install -q pandas numpy scikit-learn sentence-transformers matplotlib seaborn
!pip install -q umap-learn plotly

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import os

print("Libraries loaded")

In [ ]:
# Load the pre-trained sentence transformer model
print("Loading embedding model (this may take a minute)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
# Load datasets
candidates = pd.read_csv('../datasets/candidates.csv')
jobs = pd.read_csv('../datasets/jobs.csv')

print(f"Candidates: {len(candidates)}")
print(f"Jobs: {len(jobs)}")

# Use a subset for demonstration
sample_candidates = candidates.head(100)
sample_jobs = jobs.head(10)

print(f"Using {len(sample_candidates)} candidates and {len(sample_jobs)} jobs for embedding")

In [ ]:
# Generate embeddings for candidate resume text
print("Generating candidate embeddings...")
candidate_texts = sample_candidates['resume_text'].tolist()
candidate_embeddings = model.encode(candidate_texts, show_progress_bar=True)

print(f"Candidate embeddings shape: {candidate_embeddings.shape}")

In [ ]:
# Generate embeddings for job descriptions
print("Generating job embeddings...")
job_texts = sample_jobs['description'].tolist()
job_embeddings = model.encode(job_texts, show_progress_bar=True)

print(f"Job embeddings shape: {job_embeddings.shape}")

In [ ]:
# Calculate similarity matrix between candidates and jobs
similarity_matrix = cosine_similarity(candidate_embeddings, job_embeddings)
print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"\nSimilarity range: {similarity_matrix.min():.3f} - {similarity_matrix.max():.3f}")
print(f"Mean similarity: {similarity_matrix.mean():.3f}")

In [ ]:
# Find best matches for the first job
job_title = sample_jobs.iloc[0]['title']
job_similarities = similarity_matrix[:, 0]
top_indices = np.argsort(job_similarities)[::-1][:5]

print(f"Top 5 candidates for: {job_title}")
print("-" * 50)
for idx in top_indices:
    name = sample_candidates.iloc[idx]['name']
    score = job_similarities[idx] * 100
    print(f"  {name}: {score:.1f}%")

In [ ]:
# PCA visualization of embeddings
pca = PCA(n_components=2)
reduced = pca.fit_transform(candidate_embeddings[:50])

plt.figure(figsize=(10, 7))
plt.scatter(reduced[:, 0], reduced[:, 1], alpha=0.6, s=50, c=range(50), cmap='viridis')
plt.title('Resume Embeddings (PCA - 2D)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.colorbar(label='Candidate Index')
plt.tight_layout()
plt.show()

In [ ]:
# Save embeddings for use in ranking model
os.makedirs('../backend/app/ml_models', exist_ok=True)
np.save('../backend/app/ml_models/resume_embeddings.npy', candidate_embeddings)
print(f"Saved embeddings: {candidate_embeddings.shape}")
print("Embedding generation complete!")